In [2]:
import numpy as np
import matplotlib.pyplot as plt
import os

np.random.seed(0)

def recursive_rls(y, u, delta=1e3, lam=1.0):
    """
    Simple RLS for model y_k = a1*y_{k-1} + b0*u_{k-1} + v_k
    Parameter vector theta = [a1, b0].
    y and u are 1D arrays of length N.
    Returns final theta estimate (2,).
    """
    theta = np.zeros(2)
    P = delta * np.eye(2)
    N = len(y)
    for k in range(1, N):
        phi = np.array([-y[k-1], u[k-1]])  # shape (2,)
        denom = lam + phi @ (P @ phi)
        K = (P @ phi) / denom
        error = y[k] - phi.dot(theta)
        theta = theta + K * error
        P = (P - np.outer(K, phi) @ P) / lam
    return theta

def batch_ls(y, u):
    """
    Batch least-squares for the same model.
    Returns theta = [a1, b0].
    """
    N = len(y)
    Phi = np.column_stack([ -y[:-1], u[:-1] ])  # (N-1,2)
    y_vec = y[1:]
    # regularize tiny amount for numerical stability
    reg = 1e-12 * np.eye(2)
    theta = np.linalg.solve(Phi.T @ Phi + reg, Phi.T @ y_vec)
    return theta

# Simulation parameters
true_theta = np.array([0.707, 0.293])  # [a1, b0]
N = 1000
n_realizations = 100
sigma_v = np.sqrt(3)

thetas_batch = np.zeros((n_realizations, 2))
thetas_rls = np.zeros((n_realizations, 2))

for i in range(n_realizations):
    u = np.random.uniform(-1, 1, N)
    v = np.random.normal(0, sigma_v, N)
    y = np.zeros(N)
    for k in range(1, N):
        y[k] = -true_theta[0] * y[k-1] + true_theta[1] * u[k-1] + v[k]
    thetas_batch[i] = batch_ls(y, u)
    thetas_rls[i] = recursive_rls(y, u, delta=1e4, lam=1.0)

# Create output directory
out_dir = "./entrega/imagens"
os.makedirs(out_dir, exist_ok=True)

# Plot all histograms as subplots in a single figure
fig, axs = plt.subplots(2, 2, figsize=(10, 8), constrained_layout=True)

# a1 - batch
axs[0, 0].hist(thetas_batch[:, 0], bins=30)
axs[0, 0].axvline(true_theta[0], linestyle='--', linewidth=2)
axs[0, 0].set_title("a₁ - MQ")
axs[0, 0].set_xlabel("estimativa de a₁")
axs[0, 0].set_ylabel("contagem")

# a1 - RLS
axs[0, 1].hist(thetas_rls[:, 0], bins=30)
axs[0, 1].axvline(true_theta[0], linestyle='--', linewidth=2)
axs[0, 1].set_title("a₁ - MQR")
axs[0, 1].set_xlabel("estimativa de a₁")
axs[0, 1].set_ylabel("contagem")

# b0 - batch
axs[1, 0].hist(thetas_batch[:, 1], bins=30)
axs[1, 0].axvline(true_theta[1], linestyle='--', linewidth=2)
axs[1, 0].set_title("b₀ - MQ")
axs[1, 0].set_xlabel("estimativa de b₀")
axs[1, 0].set_ylabel("contagem")

# b0 - RLS
axs[1, 1].hist(thetas_rls[:, 1], bins=30)
axs[1, 1].axvline(true_theta[1], linestyle='--', linewidth=2)
axs[1, 1].set_title("b₀ - MQR")
axs[1, 1].set_xlabel("estimativa de b₀")
axs[1, 1].set_ylabel("contagem")

# Save single figure
file_histograms = os.path.join(out_dir, "histogramas_a.png")
fig.savefig(file_histograms)
plt.close(fig)
